## Imports

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holidays as holidays_lib
import requests

## Helper Functions

In [5]:
def _clean_ktm_data(df:pd.DataFrame, year:str) -> pd.DataFrame:
    df = df.copy()

    ## 1. Remove ridership that is negative
    df = df[df['ridership'] > 0]

    ## 2. Remove station that is 'Unknown;
    df = df[df['origin'] != 'Unknown']

    ## 3. Remove duplicates
    df = df.drop_duplicates()

    ## 4. Remove rows with missing values
    df = df.dropna()
    
    ## 5. Make sure the year is on the specified year
    
    df = df[df['date'].dt.year == int(year)]

    return df

def get_clean_ktm_data(year:int)->pd.DataFrame:
    """
    Get Official KTMB data from 'https://storage.data.gov.my/transportation/ktmb/{year}.parquet'
    """
    
    URL_DATA = f'https://storage.data.gov.my/transportation/ktmb/komuter_{year}.parquet'
    
    # Read data
    df = pd.read_parquet(URL_DATA, engine='fastparquet')
    
    # Cast datetime format
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        
    return _clean_ktm_data(df, year)

In [51]:
def get_weather_data(weather_features: list) -> pd.DataFrame:
    response = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params={
            "latitude": 3.139, "longitude": 101.6869,          # Kuala Lumpur
            "start_date": "2024-01-01", "end_date": "2024-12-31",
            "hourly": ",".join(weather_features),
            "timezone": "Asia/Kuala_Lumpur",
        },
        timeout=60,
    )
    response.raise_for_status()
    df =  pd.DataFrame(response.json()["hourly"]).rename(columns={'time':'datetime'})
    
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    return df


def get_temporal_data(year:int)->pd.DataFrame:
    YEAR_HOURS = pd.date_range(f"{year}-01-01", f"{year}-12-31 23:00", freq="h")
    
    ### First-level (base)
    my_holidays = holidays_lib.Malaysia(years=sorted(set(YEAR_HOURS.year)), subdiv="KUL")
    day_of_week = YEAR_HOURS.dayofweek.to_numpy()
    hour = YEAR_HOURS.hour.to_numpy()

    ### Second-levek (derrived from first-level) 
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)
    dow_sin = np.sin(2 * np.pi * day_of_week / 7)
    dow_cos = np.cos(2 * np.pi * day_of_week / 7)
    is_weekend = (day_of_week >= 5).astype(float)
    is_holiday = np.array([d in my_holidays for d in YEAR_HOURS.date], dtype=float)
    
    column_name = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'is_holiday']
    
    np_stack = np.stack([
        hour_sin, hour_cos, dow_sin, dow_cos, is_weekend, is_holiday,
    ], axis=1)
    
    return pd.DataFrame(np_stack, columns=column_name, index=YEAR_HOURS).reset_index().rename(columns={'index':'datetime'})

In [55]:
def join_all_df(ktm_df:pd.DataFrame, weather_df:pd.DataFrame, temporal_df:pd.DataFrame)->pd.DataFrame:
    
    selected_cols = ['datetime', 'origin', 'destination', 'ridership']
        
    ktm_df['od'] = ktm_df['origin'] + '_' + ktm_df['destination']

    ktm_df['datetime'] = ktm_df['date'] + pd.to_timedelta(ktm_df['time'] + ':00')
    
    
    ktm_df = ktm_df[selected_cols]
    
    joined_df = ktm_df.merge(
        weather_df, on='datetime', how='left'
    ).merge(
        temporal_df, on='datetime', how='left'
    )
    
    return joined_df

In [105]:
def build_station_series(
    ktm_df:pd.DataFrame,
    datetime_index:pd.DatetimeIndex,
    stations:list
) -> np.ndarray:
    
    agg_df = ktm_df.groupby([ktm_df['datetime'].dt.floor('h'), 'origin'])['ridership'].sum()
    
    # Pivot from long format to wide format
    # To fill 0 for stations without ridership at certain datetime
    # whereby other stations have data
    pivot_df = agg_df.unstack(fill_value=0) 
    
    # Reindex based on intialised value upstream (to ensure parity)
    # To fill 0 for stations that never appear as origin
    # or datetime (index) that has no data at all
    pivot_df = pivot_df.reindex(index=datetime_index, columns=stations, fill_value=0)
    
    # Convert to numpy
    numpy_df = pivot_df.to_numpy(dtype=np.float32)
    
    return numpy_df

## [1] Data Collection and Cleaning

### 1.1 KTM Data

In [6]:
ktm_2024_df = get_clean_ktm_data(2024)

In [57]:
ktm_2024_df

,date,time,origin,destination,ridership,od,datetime
0,2024-01-01,00:00,Subang Jaya,Padang Jawa,2,Subang Jaya_Padang Jawa,2024-01-01 00:00:00
1,2024-01-01,00:00,Kajang,Nilai,3,Kajang_Nilai,2024-01-01 00:00:00
2,2024-01-01,00:00,KL Sentral,Nilai,10,KL Sentral_Nilai,2024-01-01 00:00:00
3,2024-01-01,00:00,KL Sentral,Bukit Badak,6,KL Sentral_Bukit Badak,2024-01-01 00:00:00
4,2024-01-01,00:00,KL Sentral,Klang,16,KL Sentral_Klang,2024-01-01 00:00:00
...,...,...,...,...,...,...,...
2686103,2024-12-31,23:00,Midvalley,Sungai Gadut,1,Midvalley_Sungai Gadut,2024-12-31 23:00:00
2686104,2024-12-31,23:00,Subang Jaya,Padang Jawa,8,Subang Jaya_Padang Jawa,2024-12-31 23:00:00
2686105,2024-12-31,23:00,Abdullah Hukum,Padang Jawa,1,Abdullah Hukum_Padang Jawa,2024-12-31 23:00:00
2686106,2024-12-31,23:00,KL Sentral,Padang Jawa,6,KL Sentral_Padang Jawa,2024-12-31 23:00:00


In [82]:
## Derrive the stations and index in the beginning

STATIONS_LIST = sorted(ktm_2024_df['origin'].unique())
DT_INDEX = pd.date_range("2024-01-01", "2024-12-31 23:00", freq="h")

### 1.2 Weather Data

In [53]:
weather_features = [
    'temperature_2m',
    'relative_humidity_2m',
    'precipitation'
]

weather_df = get_weather_data(weather_features)

In [54]:
weather_df

,datetime,temperature_2m,relative_humidity_2m,precipitation
0,2024-01-01 00:00:00,25.6,95,0.0
1,2024-01-01 01:00:00,25.4,96,0.0
2,2024-01-01 02:00:00,25.1,96,0.0
3,2024-01-01 03:00:00,24.8,98,0.0
4,2024-01-01 04:00:00,24.7,97,0.0
...,...,...,...,...
8779,2024-12-31 19:00:00,26.1,88,0.1
8780,2024-12-31 20:00:00,25.7,93,0.0
8781,2024-12-31 21:00:00,25.9,95,0.2
8782,2024-12-31 22:00:00,25.5,96,1.0


### 1.3 Temporal Data

In [41]:
temporal_df = get_temporal_data(2024)

In [42]:
temporal_df

,datetime,hour_sin,hour_cos,dow_sin,dow_cos,is_weekend,is_holiday
0,2024-01-01 00:00:00,0.000000,1.000000,0.000000,1.00000,0.0,1.0
1,2024-01-01 01:00:00,0.258819,0.965926,0.000000,1.00000,0.0,1.0
2,2024-01-01 02:00:00,0.500000,0.866025,0.000000,1.00000,0.0,1.0
3,2024-01-01 03:00:00,0.707107,0.707107,0.000000,1.00000,0.0,1.0
4,2024-01-01 04:00:00,0.866025,0.500000,0.000000,1.00000,0.0,1.0
...,...,...,...,...,...,...,...
8779,2024-12-31 19:00:00,-0.965926,0.258819,0.781831,0.62349,0.0,0.0
8780,2024-12-31 20:00:00,-0.866025,0.500000,0.781831,0.62349,0.0,0.0
8781,2024-12-31 21:00:00,-0.707107,0.707107,0.781831,0.62349,0.0,0.0
8782,2024-12-31 22:00:00,-0.500000,0.866025,0.781831,0.62349,0.0,0.0


In [56]:
join_all_df(ktm_2024_df, weather_df, temporal_df)

,datetime,origin,destination,ridership,temperature_2m,relative_humidity_2m,precipitation,hour_sin,hour_cos,dow_sin,dow_cos,is_weekend,is_holiday
0,2024-01-01 00:00:00,Subang Jaya,Padang Jawa,2,25.6,95,0.0,0.000000,1.000000,0.000000,1.00000,0.0,1.0
1,2024-01-01 00:00:00,Kajang,Nilai,3,25.6,95,0.0,0.000000,1.000000,0.000000,1.00000,0.0,1.0
2,2024-01-01 00:00:00,KL Sentral,Nilai,10,25.6,95,0.0,0.000000,1.000000,0.000000,1.00000,0.0,1.0
3,2024-01-01 00:00:00,KL Sentral,Bukit Badak,6,25.6,95,0.0,0.000000,1.000000,0.000000,1.00000,0.0,1.0
4,2024-01-01 00:00:00,KL Sentral,Klang,16,25.6,95,0.0,0.000000,1.000000,0.000000,1.00000,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2674284,2024-12-31 23:00:00,Midvalley,Sungai Gadut,1,24.9,98,0.0,-0.258819,0.965926,0.781831,0.62349,0.0,0.0
2674285,2024-12-31 23:00:00,Subang Jaya,Padang Jawa,8,24.9,98,0.0,-0.258819,0.965926,0.781831,0.62349,0.0,0.0
2674286,2024-12-31 23:00:00,Abdullah Hukum,Padang Jawa,1,24.9,98,0.0,-0.258819,0.965926,0.781831,0.62349,0.0,0.0
2674287,2024-12-31 23:00:00,KL Sentral,Padang Jawa,6,24.9,98,0.0,-0.258819,0.965926,0.781831,0.62349,0.0,0.0


## [2] Data Preprocessing

### 2.1 Data Cleaning

### 2.2 Data Transformation and Integration

### 2.3 Feature Engineering

### 2.4 Graph Structure

In [ ]:
hour_origin_ridership_df = ktm_2024_df.groupby([ktm_2024_df['datetime'].dt.floor("h"), "origin"])['ridership'].sum().reset_index()

In [79]:
(
    ktm_2024_df.groupby([ktm_2024_df['datetime'].dt.floor("h"), "origin"])['ridership'].sum().reset_index()
)


,datetime,origin,ridership
0,2024-01-01 00:00:00,Bandar Tasek Selatan,5
1,2024-01-01 00:00:00,Bangi,1
2,2024-01-01 00:00:00,Batu Tiga,1
3,2024-01-01 00:00:00,KL Sentral,56
4,2024-01-01 00:00:00,Kajang,9
...,...,...,...
367030,2024-12-31 23:00:00,Subang Jaya,14
367031,2024-12-31 23:00:00,Sungai Buloh,19
367032,2024-12-31 23:00:00,Taman Wahyu,1
367033,2024-12-31 23:00:00,Tanjong Malim,3


In [85]:
(
    ktm_2024_df.groupby([ktm_2024_df['datetime'].dt.floor("h"), "origin"])['ridership'].sum()
    .unstack(fill_value=0)
)


origin,Abdullah Hukum,Angkasapuri,Bandar Tasek Selatan,Bangi,Bank Negara,Batang Benar,Batang Kali,Batu Caves,Batu Kentonmen,Batu Tiga,...,Shah Alam,Subang Jaya,Sungai Buloh,Sungai Gadut,Taman Wahyu,Tanjong Malim,Telok Gadong,Telok Pulai,Tiroi,UKM
datetime,,,,,,,,,,,,,,,,,,,,,
2024-01-01 00:00:00,0,0,5,1,0,0,0,0,0,1,...,0,13,37,0,1,10,0,0,0,1
2024-01-01 01:00:00,0,0,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2024-01-01 02:00:00,0,0,3,0,1,0,1,20,0,1,...,0,2,1,4,0,3,1,0,0,0
2024-01-01 03:00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2024-01-01 04:00:00,6,0,89,38,4,30,8,23,1,10,...,33,23,36,15,3,128,15,7,7,65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 19:00:00,107,3,150,45,114,24,18,77,3,69,...,58,456,216,24,9,45,38,22,4,64
2024-12-31 20:00:00,33,0,65,24,46,46,17,66,1,21,...,29,113,107,7,15,108,19,11,0,35
2024-12-31 21:00:00,24,1,63,41,15,68,12,44,2,24,...,27,68,61,11,19,48,24,5,8,19


In [84]:
(
    ktm_2024_df.groupby([ktm_2024_df['datetime'].dt.floor("h"), "origin"])['ridership'].sum()
    .unstack(fill_value=0)
    .reindex(index=DT_INDEX, columns=STATIONS_LIST, fill_value=0)
)

origin,Abdullah Hukum,Angkasapuri,Bandar Tasek Selatan,Bangi,Bank Negara,Batang Benar,Batang Kali,Batu Caves,Batu Kentonmen,Batu Tiga,...,Shah Alam,Subang Jaya,Sungai Buloh,Sungai Gadut,Taman Wahyu,Tanjong Malim,Telok Gadong,Telok Pulai,Tiroi,UKM
2024-01-01 00:00:00,0,0,5,1,0,0,0,0,0,1,...,0,13,37,0,1,10,0,0,0,1
2024-01-01 01:00:00,0,0,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2024-01-01 02:00:00,0,0,3,0,1,0,1,20,0,1,...,0,2,1,4,0,3,1,0,0,0
2024-01-01 03:00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2024-01-01 04:00:00,6,0,89,38,4,30,8,23,1,10,...,33,23,36,15,3,128,15,7,7,65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 19:00:00,107,3,150,45,114,24,18,77,3,69,...,58,456,216,24,9,45,38,22,4,64
2024-12-31 20:00:00,33,0,65,24,46,46,17,66,1,21,...,29,113,107,7,15,108,19,11,0,35
2024-12-31 21:00:00,24,1,63,41,15,68,12,44,2,24,...,27,68,61,11,19,48,24,5,8,19
2024-12-31 22:00:00,21,4,40,13,13,32,7,12,0,18,...,31,61,24,4,9,42,4,5,1,9


In [ ]:
(
    ktm_2024_df.groupby([ktm_2024_df['datetime'].dt.floor("h"), "origin"])['ridership'].sum()
    .unstack(fill_value=0) # To fill gaps in the observed data
    .reindex(index=DT_INDEX, columns=STATIONS_LIST, fill_value=0) # To fill rows and columns that were never present in the data
).to_numpy(dtype=np.float32)

array([[ 0.,  0.,  5., ...,  0.,  0.,  1.],
       [ 0.,  0.,  1., ...,  0.,  0.,  1.],
       [ 0.,  0.,  3., ...,  0.,  0.,  0.],
       ...,
       [24.,  1., 63., ...,  5.,  8., 19.],
       [21.,  4., 40., ...,  5.,  1.,  9.],
       [25.,  0., 25., ...,  0.,  0.,  4.]], dtype=float32)

In [87]:
test = (
    ktm_2024_df.groupby([ktm_2024_df['datetime'].dt.floor("h"), "origin"])['ridership'].sum()
    .unstack(fill_value=0) # To fill gaps in the observed data
    .reindex(index=DT_INDEX, columns=STATIONS_LIST, fill_value=0) # To fill rows and columns that were never present in the data
).to_numpy(dtype=np.float32)

In [93]:
test.shape

(8784, 57)

In [104]:
test[:, STATIONS_LIST.index('Nilai')].shape

(8784,)

In [66]:
hour_origin_ridership_df

,datetime,origin,ridership
0,2024-01-01 00:00:00,Bandar Tasek Selatan,5
1,2024-01-01 00:00:00,Bangi,1
2,2024-01-01 00:00:00,Batu Tiga,1
3,2024-01-01 00:00:00,KL Sentral,56
4,2024-01-01 00:00:00,Kajang,9
...,...,...,...
367030,2024-12-31 23:00:00,Subang Jaya,14
367031,2024-12-31 23:00:00,Sungai Buloh,19
367032,2024-12-31 23:00:00,Taman Wahyu,1
367033,2024-12-31 23:00:00,Tanjong Malim,3


## [3] Model Development

### Train-test-validation Split

### M1 Variation

### M2 Variation

### M3 Variation

## [4] Model Evaluation